# 02 - Build Span-Completion Profile Tokenizer

Build the profile-aware tokenizer artifacts for span completion at `data/compiled/refseq_bacteria_span_completion_profile_pretrain/tokenizer_map.json`. This notebook reuses the root stage-1 protein `tokenizer_map.json` as the sequence tokenizer and does not train a new protein tokenizer.


In [ ]:
from __future__ import annotations

import json
import os
import sys
from itertools import islice
from pathlib import Path

import yaml


def find_repo_dir_for_import(start: Path) -> Path:
    candidates: list[Path] = []
    env_value = os.environ.get("MDNAC_REPO_DIR")
    if env_value:
        candidates.append(Path(env_value).expanduser())
    resolved_start = start.expanduser().resolve()
    candidates.extend([resolved_start, *resolved_start.parents])
    for candidate in candidates:
        resolved = candidate.expanduser().resolve()
        if (resolved / "pyproject.toml").exists() and (resolved / "libs").is_dir():
            return resolved
    raise RuntimeError("Could not locate MDNAC repo. Run inside the repo or set MDNAC_REPO_DIR.")


REPO_DIR = find_repo_dir_for_import(Path.cwd())
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from libs.notebook_runtime import apply_instruction_notebook_overrides, bootstrap_notebook
from libs.instruction.trainer import build_instruction_training_config

RUNTIME = bootstrap_notebook(REPO_DIR)
REPO_DIR = Path(RUNTIME["repo_dir"])
print(json.dumps(RUNTIME, indent=2, ensure_ascii=False))


In [ ]:
CONFIG_PATH = Path(os.environ.get("MDNAC_PROFILE_SPAN_CONFIG", REPO_DIR / "config" / "profile_span_completion.yaml")).expanduser()
if not CONFIG_PATH.is_absolute():
    CONFIG_PATH = (REPO_DIR / CONFIG_PATH).resolve()

raw_config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8")) or {}
CONFIG = build_instruction_training_config(REPO_DIR, raw_config)
CONFIG = apply_instruction_notebook_overrides(CONFIG)

instruction_paths = tuple(Path(path).expanduser().resolve() for path in CONFIG.instruction_jsonl)
protein_tokenizer_map = Path(CONFIG.sequence_tokenizer_map_path).expanduser().resolve()
expected_protein_tokenizer_map = (REPO_DIR / "tokenizer_map.json").resolve()
artifact_dir = Path(CONFIG.artifact_dir).expanduser().resolve()
artifact_train_txt = artifact_dir / "train.txt"
artifact_tokenizer_map = artifact_dir / "tokenizer_map.json"
source_stats_path = Path(CONFIG.artifact_source_jsonl).expanduser().resolve() / "stats.json"

missing_instruction_paths = [path for path in instruction_paths if not path.is_file()]
assert not missing_instruction_paths, f"Missing instruction JSONL part: {missing_instruction_paths[0]}"
assert protein_tokenizer_map.is_file(), f"Missing stage-1 protein tokenizer map: {protein_tokenizer_map}"
assert protein_tokenizer_map == expected_protein_tokenizer_map, (
    "This span-completion artifact must reuse the root stage-1 protein tokenizer map. "
    f"Expected {expected_protein_tokenizer_map}, got {protein_tokenizer_map}."
)
assert CONFIG.default_sequence_type == "protein"
assert CONFIG.prompt_format == "compact_profile"

summary = {
    "config_path": str(CONFIG_PATH),
    "instruction_part_count": len(instruction_paths),
    "instruction_paths": [str(path) for path in instruction_paths],
    "source_stats": str(source_stats_path) if source_stats_path.is_file() else None,
    "artifact_dir": str(artifact_dir),
    "artifact_tokenizer_map": str(artifact_tokenizer_map),
    "profile_vocab_size": CONFIG.profile_vocab_size,
    "profile_sample_size": CONFIG.artifact_profile_sample_size,
    "sequence_tokenizer_map_path": str(protein_tokenizer_map),
    "auto_detect_sequence_tokenizer_map": CONFIG.auto_detect_sequence_tokenizer_map,
    "rebuild_artifacts": CONFIG.rebuild_artifacts,
}
print(json.dumps(summary, indent=2, ensure_ascii=False))


In [ ]:
from libs.instruction.schema import iter_instruction_records

records = tuple(islice(
    iter_instruction_records(
        instruction_paths,
        default_sequence_type=CONFIG.default_sequence_type,
        instruction_field=CONFIG.instruction_field,
        input_field=CONFIG.input_field,
        output_field=CONFIG.output_field,
        prompt_format=CONFIG.prompt_format,
    ),
    CONFIG.artifact_profile_sample_size,
))
if not records:
    raise ValueError(f"No valid instruction records found in {instruction_paths}")

sequence_types = sorted({record.sequence_type for record in records})
assert sequence_types == ["protein"], f"Expected only protein records, got {sequence_types}"

sample_report = {
    "sampled_records": len(records),
    "sequence_types": sequence_types,
    "first_profile_preview": records[0].profile[:320],
    "first_target_length": len(records[0].sequence),
}
print(json.dumps(sample_report, indent=2, ensure_ascii=False))


In [ ]:
from libs.core.pretrain.profiled import (
    MDCProfileSequencePretrainArtifacts,
    save_mdc_profile_sequence_pretrain_artifacts,
)

if CONFIG.rebuild_artifacts or not artifact_train_txt.is_file() or not artifact_tokenizer_map.is_file():
    artifact_summary = save_mdc_profile_sequence_pretrain_artifacts(
        records,
        artifact_dir,
        sequence_type=CONFIG.default_sequence_type,
        kmer_size=CONFIG.kmer_size,
        profile_vocab_size=CONFIG.profile_vocab_size,
        sequence_tokenizer_map_path=protein_tokenizer_map,
    )
    print("Built artifacts:", json.dumps(artifact_summary.__dict__, indent=2, ensure_ascii=False))
else:
    print(f"Reusing existing artifacts under {artifact_dir}")

artifacts = MDCProfileSequencePretrainArtifacts.from_directory(artifact_dir)
tokenizer_payload = json.loads(artifact_tokenizer_map.read_text(encoding="utf-8"))
sequence_tokenizer_payload = tokenizer_payload["sequence_tokenizer"]
source_tokenizer_map_path = sequence_tokenizer_payload.get("source_tokenizer_map_path")

assert artifact_train_txt.is_file(), f"Missing artifact train.txt: {artifact_train_txt}"
assert artifact_tokenizer_map.is_file(), f"Missing artifact tokenizer_map.json: {artifact_tokenizer_map}"
assert sequence_tokenizer_payload["type"] == "sequence_bpe"
assert tokenizer_payload["compatibility"]["sequence_token_ids_preserved"] is True
assert source_tokenizer_map_path, "Profile-aware artifact did not record the source protein tokenizer map."
assert Path(source_tokenizer_map_path).resolve() == protein_tokenizer_map

profile_vocab = tokenizer_payload["profile_tokenizer"]["tokenizer"]["str_to_int"]
mask_related_tokens = sorted(
    token for token in profile_vocab
    if "MASK" in token or "mask_" in token or "partial_sequence" in token
)

artifact_report = {
    "train_txt": str(artifact_train_txt),
    "tokenizer_map": str(artifact_tokenizer_map),
    "record_count": artifacts.record_count,
    "sequence_type": artifacts.sequence_type,
    "sequence_tokenizer_type": artifacts.sequence_tokenizer_type,
    "sequence_vocab_size": artifacts.sequence_vocab_size,
    "profile_vocab_size": artifacts.profile_vocab_size,
    "protein_tokenizer_loaded_from": source_tokenizer_map_path,
    "mask_related_profile_tokens_preview": mask_related_tokens[:24],
    "mask_related_profile_token_count": len(mask_related_tokens),
}
print(json.dumps(artifact_report, indent=2, ensure_ascii=False))


In [ ]:
encoded = artifacts.encode_record(records[0])
fused = artifacts.build_fused_batch([encoded])
sequence_start, sequence_end = [int(value) for value in fused.sequence_spans[0].tolist()]
sequence_ids = artifacts.sequence_tokenizer.encode(records[0].sequence)
protein_vocab_rows = int(tokenizer_payload["compatibility"]["protein_checkpoint_vocab_rows"])

assert sequence_ids
assert all(0 <= token_id < protein_vocab_rows for token_id in sequence_ids)

smoke_report = {
    "profile_token_count": len(encoded.profile_token_ids),
    "sequence_token_count": len(encoded.sequence_token_ids),
    "sequence_span": [sequence_start, sequence_end],
    "protein_vocab_rows": protein_vocab_rows,
    "max_sequence_token_id": max(sequence_ids),
    "decoded_target_matches": artifacts.sequence_tokenizer.decode(sequence_ids) == records[0].sequence,
}
print(json.dumps(smoke_report, indent=2, ensure_ascii=False))
